# Selection Stage 1 — SHAP, permutation importance, and readout-count stability

**Feature engineering is paused.** This notebook reads the completed nested-selection result. It never fits the selection study itself.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs/selection_stage1.json').is_file())
for p in (ROOT, ROOT/'src'):
    if str(p) not in sys.path: sys.path.insert(0,str(p))
from scripts.selection_stage1 import figures, write_dashboard
result=json.loads((ROOT/'reports/selection_stage1/results.json').read_text())
CHARTS=figures(result)
print('Status:',result['status']); print('Shortlist:',result['shortlist'])

Status: SELECTION_STAGE1_COMPLETE
Shortlist: ['add_relations']


## Metric discipline

Macro AUC is primary. Brier and probability RMSE provide the same squared-error ordering, so the count rule uses an AUC one-SE set with a Brier one-SE guard rather than treating RMSE as an independent objective.

In [2]:
display(pd.DataFrame(result['outer_folds'])[['outer_fold','selected_count','macro_auc','advertising_auc','legal_advice_auc','pooled_auc','brier','probability_rmse','log_loss']])
CHARTS[0].show(renderer='plotly_mimetype'); CHARTS[1].show(renderer='plotly_mimetype')

,outer_fold,selected_count,macro_auc,advertising_auc,legal_advice_auc,pooled_auc,brier,probability_rmse,log_loss
0,0,3,0.727971,0.709259,0.746683,0.734917,0.209219,0.457405,0.602350
1,1,3,0.670562,0.596154,0.744970,0.712343,0.223075,0.472309,0.640705
2,2,1,0.734539,0.692593,0.776485,0.756863,0.203644,0.451270,0.592452
3,3,1,0.756914,0.759259,0.754568,0.753445,0.197443,0.444345,0.583935
4,4,1,0.726594,0.686111,0.767076,0.744752,0.210930,0.459271,0.611453


## SHAP + held-out permutation importance

For the linear meta-model, SHAP is exact under the independent-background linear formulation. Permutation importance is measured with the registered macro-AUC metric on held-out inner folds. Correlated readouts are clustered before count selection.

In [3]:
display(pd.DataFrame(result['importance']).head(20))
CHARTS[2].show(renderer='plotly_mimetype'); CHARTS[3].show(renderer='plotly_mimetype')

,mean_abs_linear_shap,mean_permutation_macro_auc_drop,readout,selection_frequency
0,0.544175,0.105025,add_relations,0.6
1,0.407792,0.075555,add_legacy_support,0.4
2,0.386439,0.073493,without_windows,0.4
3,0.528029,0.101097,frozen_margin_all,0.2
4,0.430714,0.087355,without_bm25,0.2
5,0.349098,0.065638,add_scope,0.0
6,0.357556,0.063520,pair_bm25_passages,0.0
7,0.332105,0.058125,add_local_density,0.0
8,0.315259,0.048593,qwen_raw,0.0
9,0.264209,0.044969,add_matched_pairs,0.0


## Stability and selected count

Each outer fold owns its entire inner ranking and count-selection process. The final shortlist is based on cross-fold selection frequency and the median selected count; outer-test performance is not used to pick a readout for that fold.

In [4]:
CHARTS[4].show(renderer='plotly_mimetype'); CHARTS[5].show(renderer='plotly_mimetype')

## Probability quality and Stage-2 handoff

Stage 1 is readout/family-level selection, not raw token-column selection. Stage 2 will perform model-specific feature selection inside its nested training folds before comparing models.

In [5]:
CHARTS[6].show(renderer='plotly_mimetype'); CHARTS[7].show(renderer='plotly_mimetype')
print('Dashboard:',write_dashboard(ROOT,result))
for item in result['limitations']: print('-',item)

Dashboard: /home/sagemaker-user/projects/jigsaw-rule-classifier/reports/selection_stage1/dashboard.html
- Nested resampling occurs on a repeatedly inspected development cohort, not a fresh holdout.
- Historical feature invention already adapted to these policies.
- Selection is over saved model/readout channels; raw token-column pruning belongs inside Stage 2 model pipelines.
- Interventional linear SHAP assumes an independent background; correlated channels are clustered and held-out macro-AUC permutation importance is also required.
- Probability RMSE equals sqrt(Brier) and is a secondary probability-quality guard, not an independent objective.
- No feature, model, ensemble, or Kaggle submission is promoted automatically.
